# ConvNeXt cattle model: PTH → ONNX, then test on GitHub images

1. **Cell 1-2:** converts `convnext_cattle_best.pth` (in `Cow_lumpy_skin/runs_convnext`) to `convnext_cattle_best.onnx` in the same folder.
2. **Cell 3:** downloads the test images from the two GitHub folders (`Lumpy Skin`, `Normal Skin`) and runs the ONNX model on them (CPU).

A **CPU runtime is enough** for this notebook (`Runtime → Change runtime type → None/CPU`), which saves your free GPU quota. Every test image is resized to 640×640 **in memory only**.

In [ ]:
!pip install -q timm onnx onnxruntime

## 1. Convert `.pth` → `.onnx`

In [ ]:
import os, json, shutil
import numpy as np
import torch, timm
from google.colab import drive
drive.mount('/content/drive')

os.chdir('/content/drive/MyDrive/Cow_lumpy_skin/runs_convnext')

# ---- EDIT IF NEEDED ----
PTH_NAME  = 'convnext_cattle_best.pth'
ONNX_NAME = 'convnext_cattle_best.onnx'
OPSET     = 17
# ------------------------

def load_checkpoint(path):
    try:
        return torch.load(path, map_location='cpu', weights_only=True)
    except Exception:
        return torch.load(path, map_location='cpu', weights_only=False)   # your own file only

ck = load_checkpoint(PTH_NAME)
state       = ck['model'] if 'model' in ck else ck
class_names = ck.get('class_names') or ['Class_1', 'Class_2']
model_name  = ck.get('model_name') or 'convnext_base.fb_in22k_ft_in1k'
size        = ck.get('img_size') or 640
if 'epoch' in ck:
    print(f"Checkpoint from epoch {ck['epoch'] + 1} | best val loss so far: {ck.get('best_val_loss')}")

model = timm.create_model(model_name, pretrained=False, num_classes=len(class_names))
model.load_state_dict(state)
model.eval()
del ck, state
print(f"Rebuilt {model_name} | classes: {class_names} | input: {size}x{size}")

# Export to fast LOCAL disk first, then copy into Drive (one in-place overwrite, no temp files on Drive)
LOCAL_ONNX = '/content/' + ONNX_NAME
dummy = torch.randn(1, 3, size, size)
kwargs = dict(input_names=['input'], output_names=['logits'],
              dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},
              opset_version=OPSET, do_constant_folding=True)
print("Exporting to ONNX (1-3 minutes on CPU) ...")
with torch.no_grad():
    try:
        torch.onnx.export(model, dummy, LOCAL_ONNX, dynamo=False, **kwargs)
    except TypeError:
        torch.onnx.export(model, dummy, LOCAL_ONNX, **kwargs)

# Embed class names / size / normalization inside the ONNX file
import onnx
m = onnx.load(LOCAL_ONNX)
for k, v in {'class_names': json.dumps(class_names), 'img_size': str(size),
             'mean': '0.485,0.456,0.406', 'std': '0.229,0.224,0.225', 'model_name': model_name}.items():
    entry = m.metadata_props.add(); entry.key, entry.value = k, v
onnx.checker.check_model(m)
onnx.save(m, LOCAL_ONNX)
del m

# Verify: PyTorch vs ONNX Runtime on the same random input
import onnxruntime as ort
x = np.random.RandomState(0).randn(1, 3, size, size).astype(np.float32)
with torch.no_grad():
    ref = model(torch.from_numpy(x)).numpy()
sess = ort.InferenceSession(LOCAL_ONNX, providers=['CPUExecutionProvider'])
got = sess.run(None, {sess.get_inputs()[0].name: x})[0]
diff = float(np.abs(ref - got).max())
print(f"Max |PyTorch - ONNX| logit difference: {diff:.2e} ({'OK' if diff < 1e-3 else 'larger than expected, check export'})")

shutil.copyfile(LOCAL_ONNX, os.path.join(os.getcwd(), ONNX_NAME))
print(f"Saved to Drive: {os.path.join(os.getcwd(), ONNX_NAME)} ({os.path.getsize(LOCAL_ONNX)/1024**2:.0f} MB)")
del model, sess

## 2. Test the ONNX model on the GitHub test images
Downloads the images from both GitHub folders (via the GitHub API), runs the ONNX model on CPU, and shows predictions.

**Set `FOLDER_TRUE_CLASS`** to tell the notebook which class each GitHub folder represents (e.g. `'Class_2'`), so accuracy can be computed. If you leave it as `None`, you still get a table of how many images from each folder were predicted as each class. Your training folders were named `Class_1` / `Class_2`, so the mapping to *Lumpy Skin* / *Normal Skin* is yours to fill in.

In [ ]:
import os, re, json, time, urllib.parse
import requests
import numpy as np
import pandas as pd
import onnxruntime as ort
import matplotlib.pyplot as plt
from PIL import Image

ONNX_PATH = '/content/drive/MyDrive/Cow_lumpy_skin/runs_convnext/convnext_cattle_best.onnx'

GITHUB_FOLDERS = [
    'https://github.com/Diagopreutic/Cattle_lumpy_skin_detection/tree/main/test_images/Lumpy%20Skin',
    'https://github.com/Diagopreutic/Cattle_lumpy_skin_detection/tree/main/test_images/Normal%20Skin',
]
# Which model class does each GitHub folder correspond to? e.g. 'Class_2'. None = unknown.
FOLDER_TRUE_CLASS = {'Lumpy Skin': None, 'Normal Skin': None}

EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
DEST_ROOT = '/content/test_images'
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
RESAMPLE = getattr(Image, 'Resampling', Image).BILINEAR

# ---------- 1) download the images ----------
def download_folder(url):
    m = re.match(r'https://github\.com/([^/]+)/([^/]+)/tree/([^/]+)/(.+)$', url.rstrip('/'))
    assert m, f"Not a GitHub folder URL: {url}"
    owner, repo, ref, path = m.groups()
    path = urllib.parse.unquote(path)
    api = f'https://api.github.com/repos/{owner}/{repo}/contents/{urllib.parse.quote(path)}'
    r = requests.get(api, params={'ref': ref}, timeout=30)
    if r.status_code != 200:
        raise RuntimeError(f"GitHub API returned {r.status_code} for {path}. "
                           f"Check that the repo/folder is public and the URL is right. {r.text[:200]}")
    items = [i for i in r.json() if i.get('type') == 'file' and i['name'].lower().endswith(EXTS)]
    folder = os.path.basename(path)
    out_dir = os.path.join(DEST_ROOT, folder)
    os.makedirs(out_dir, exist_ok=True)
    for it in items:
        dst = os.path.join(out_dir, it['name'])
        if not os.path.exists(dst):
            with open(dst, 'wb') as f:
                f.write(requests.get(it['download_url'], timeout=60).content)
    print(f"{folder}: {len(items)} image(s) -> {out_dir}")
    return folder, out_dir

folders = dict(download_folder(u) for u in GITHUB_FOLDERS)   # {'Lumpy Skin': dir, 'Normal Skin': dir}

# ---------- 2) load the ONNX model ----------
sess = ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
inp = sess.get_inputs()[0]
meta = sess.get_modelmeta().custom_metadata_map
class_names = json.loads(meta['class_names']) if 'class_names' in meta else ['Class_1', 'Class_2']
size = inp.shape[2] if isinstance(inp.shape[2], int) else int(meta.get('img_size', 640))
print(f"\nModel loaded | classes: {class_names} | input: {size}x{size} | CPU")

def preprocess(path):
    with Image.open(path) as im:                      # original file is never modified
        im = im.convert('RGB').resize((size, size), RESAMPLE)
    a = (np.asarray(im, dtype=np.float32) / 255.0 - MEAN) / STD
    return np.ascontiguousarray(a.transpose(2, 0, 1)[None], dtype=np.float32)

def softmax(z):
    e = np.exp(z - z.max()); return e / e.sum()

# ---------- 3) run inference ----------
rows, t0 = [], time.time()
for folder, d in folders.items():
    for fn in sorted(os.listdir(d)):
        if not fn.lower().endswith(EXTS): continue
        p = os.path.join(d, fn)
        probs = softmax(sess.run(None, {inp.name: preprocess(p)})[0][0])
        k = int(probs.argmax())
        rows.append(dict(folder=folder, file=fn, path=p, pred=class_names[k], conf=float(probs[k]),
                         **{f'p_{c}': float(pr) for c, pr in zip(class_names, probs)}))
        print(f"{folder:12s} | {fn[:40]:40s} -> {class_names[k]} ({probs[k]*100:.1f}%)")
print(f"\n{len(rows)} images in {time.time() - t0:.1f}s")

df = pd.DataFrame(rows)
print("\nPredictions per GitHub folder:")
display(pd.crosstab(df['folder'], df['pred']))

# accuracy, if you told the notebook which class each folder stands for
known = df[df['folder'].map(FOLDER_TRUE_CLASS).notna()]
if len(known):
    truth = known['folder'].map(FOLDER_TRUE_CLASS)
    acc = (known['pred'] == truth).mean()
    print(f"Accuracy on {len(known)} images: {acc:.4f}")
    wrong = known[known['pred'] != truth]
    if len(wrong): display(wrong[['folder', 'file', 'pred', 'conf']])
else:
    print("Set FOLDER_TRUE_CLASS above to also get accuracy.")

df.drop(columns='path').to_csv('/content/drive/MyDrive/Cow_lumpy_skin/runs_convnext/github_test_results.csv', index=False)

# ---------- 4) preview ----------
sample = df.sample(min(12, len(df)), random_state=0)
cols = 4; rws = int(np.ceil(len(sample) / cols))
fig, axs = plt.subplots(rws, cols, figsize=(4 * cols, 3.6 * rws))
for ax in np.atleast_1d(axs).ravel(): ax.axis('off')
for ax, (_, r) in zip(np.atleast_1d(axs).ravel(), sample.iterrows()):
    ax.imshow(Image.open(r['path']).convert('RGB'))
    ax.set_title(f"{r['folder']}\n-> {r['pred']} ({r['conf']*100:.0f}%)", fontsize=10)
plt.tight_layout(); plt.show()

## 3. Same test, using the ONNX model hosted on GitHub
Downloads `convnext_cattle_best.onnx` from the GitHub repo and tests the same GitHub images. Everything stays **in memory**: the model and images are never written to disk, and results are only printed (no CSV or other file). This cell is self-contained, so it does not need cells 1-2, and it needs no Drive access.

In [ ]:
import io, re, json, time, urllib.parse
import requests
import numpy as np
import pandas as pd
import onnxruntime as ort
import matplotlib.pyplot as plt
from PIL import Image

GITHUB_ONNX_URL = 'https://github.com/Diagopreutic/Cattle_lumpy_skin_detection/blob/main/convnext_cattle_best.onnx'
GITHUB_FOLDERS = [
    'https://github.com/Diagopreutic/Cattle_lumpy_skin_detection/tree/main/test_images/Lumpy%20Skin',
    'https://github.com/Diagopreutic/Cattle_lumpy_skin_detection/tree/main/test_images/Normal%20Skin',
]
# Which model class does each GitHub folder correspond to? e.g. 'Class_2'. None = unknown.
FOLDER_TRUE_CLASS = {'Lumpy Skin': None, 'Normal Skin': None}

EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
RESAMPLE = getattr(Image, 'Resampling', Image).BILINEAR

# ---------- 1) download the ONNX model into memory ----------
raw_url = GITHUB_ONNX_URL.replace('/blob/', '/raw/', 1)      # the /raw/ form also serves Git LFS files
print("Downloading ONNX model ...")
r = requests.get(raw_url, stream=True, timeout=60)
if r.status_code != 200:
    raise RuntimeError(f"Could not download the ONNX file (HTTP {r.status_code}). Is the repo public and the path correct?")
buf, done, t0 = io.BytesIO(), 0, time.time()
for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
    buf.write(chunk); done += len(chunk)
    print(f"\r  {done / 1024**2:.0f} MB", end='')
onnx_bytes = buf.getvalue(); del buf
print(f"\nDownloaded {len(onnx_bytes) / 1024**2:.0f} MB in {time.time() - t0:.0f}s")
if onnx_bytes[:40].startswith(b'version https://git-lfs') or len(onnx_bytes) < 1024 * 1024:
    raise RuntimeError("Got a Git LFS pointer or a tiny file instead of the real model. "
                       "Check that the ONNX file in the repo is the full model.")

sess = ort.InferenceSession(onnx_bytes, providers=['CPUExecutionProvider'])
del onnx_bytes
inp = sess.get_inputs()[0]
meta = sess.get_modelmeta().custom_metadata_map
class_names = json.loads(meta['class_names']) if 'class_names' in meta else ['Class_1', 'Class_2']
size = inp.shape[2] if isinstance(inp.shape[2], int) else int(meta.get('img_size', 640))
print(f"Model loaded | classes: {class_names} | input: {size}x{size} | CPU")

# ---------- 2) download the test images into memory ----------
def fetch_folder(url):
    m = re.match(r'https://github\.com/([^/]+)/([^/]+)/tree/([^/]+)/(.+)$', url.rstrip('/'))
    assert m, f"Not a GitHub folder URL: {url}"
    owner, repo, ref, path = m.groups()
    path = urllib.parse.unquote(path)
    api = f'https://api.github.com/repos/{owner}/{repo}/contents/{urllib.parse.quote(path)}'
    resp = requests.get(api, params={'ref': ref}, timeout=30)
    if resp.status_code != 200:
        raise RuntimeError(f"GitHub API returned {resp.status_code} for {path}. {resp.text[:200]}")
    items = [i for i in resp.json() if i.get('type') == 'file' and i['name'].lower().endswith(EXTS)]
    images = [(i['name'], requests.get(i['download_url'], timeout=60).content) for i in items]
    print(f"{os.path.basename(path)}: {len(images)} image(s)")
    return os.path.basename(path), images

import os
data = dict(fetch_folder(u) for u in GITHUB_FOLDERS)     # {'Lumpy Skin': [(name, bytes), ...], ...}

def preprocess(img_bytes):
    with Image.open(io.BytesIO(img_bytes)) as im:         # resized in memory only
        im = im.convert('RGB').resize((size, size), RESAMPLE)
    a = (np.asarray(im, dtype=np.float32) / 255.0 - MEAN) / STD
    return np.ascontiguousarray(a.transpose(2, 0, 1)[None], dtype=np.float32)

def softmax(z):
    e = np.exp(z - z.max()); return e / e.sum()

# ---------- 3) run inference and print ----------
rows, t0 = [], time.time()
for folder, images in data.items():
    for name, b in images:
        probs = softmax(sess.run(None, {inp.name: preprocess(b)})[0][0])
        k = int(probs.argmax())
        rows.append(dict(folder=folder, file=name, pred=class_names[k], conf=float(probs[k]), raw=b,
                         **{f'p_{c}': float(pr) for c, pr in zip(class_names, probs)}))
        print(f"{folder:12s} | {name[:40]:40s} -> {class_names[k]} ({probs[k]*100:.1f}%)")
print(f"\n{len(rows)} images in {time.time() - t0:.1f}s")

df = pd.DataFrame(rows)
print("\nPredictions per GitHub folder:")
display(pd.crosstab(df['folder'], df['pred']))

known = df[df['folder'].map(FOLDER_TRUE_CLASS).notna()]
if len(known):
    truth = known['folder'].map(FOLDER_TRUE_CLASS)
    print(f"Accuracy on {len(known)} images: {(known['pred'] == truth).mean():.4f}")
    wrong = known[known['pred'] != truth]
    if len(wrong): display(wrong[['folder', 'file', 'pred', 'conf']])
else:
    print("Set FOLDER_TRUE_CLASS above to also get accuracy.")

# ---------- 4) preview (shown only, not saved) ----------
sample = df.sample(min(12, len(df)), random_state=0)
cols = 4; rws = int(np.ceil(len(sample) / cols))
fig, axs = plt.subplots(rws, cols, figsize=(4 * cols, 3.6 * rws))
for ax in np.atleast_1d(axs).ravel(): ax.axis('off')
for ax, (_, r_) in zip(np.atleast_1d(axs).ravel(), sample.iterrows()):
    ax.imshow(Image.open(io.BytesIO(r_['raw'])).convert('RGB'))
    ax.set_title(f"{r_['folder']}\n-> {r_['pred']} ({r_['conf']*100:.0f}%)", fontsize=10)
plt.tight_layout(); plt.show()